# Notebook Haxball Headless Host

Notebook para conectar con Haxball Headless Host, registrar jugadores por equipo, resultados de partidos y persistir la informacion.

Antes de ejecutar:
- Define `HAXBALL_TOKEN` en tu entorno.
- Asegurate de tener `npm install` ejecutado en este proyecto.
- Este notebook asume que los datos quedan en `./data`.

## 1. Configurar el entorno y dependencias
Instalar y cargar las librerias necesarias para conexion web, captura de eventos y persistencia de datos.

In [3]:
from pathlib import Path
import os
import json
import time
import subprocess
import pandas as pd

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
EVENTS_FILE = DATA_DIR / "events.jsonl"
MATCHES_FILE = DATA_DIR / "matches.jsonl"
PLAYERS_FILE = DATA_DIR / "players.json"

DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Proyecto: {PROJECT_DIR}")
print(f"Data dir: {DATA_DIR}")

Proyecto: /home/andreschirinos/Proyectos/haxball-host
Data dir: /home/andreschirinos/Proyectos/haxball-host/data


In [4]:
if not (PROJECT_DIR / "node_modules").exists():
    print("Instalando dependencias npm...")
    install = subprocess.run(["npm", "install"], capture_output=True, text=True)
    print(install.stdout[-3000:])
    if install.returncode != 0:
        print(install.stderr[-3000:])
        raise RuntimeError("npm install fallo")
else:
    print("Dependencias npm ya instaladas")

Dependencias npm ya instaladas


## 2. Conectar con Haxball Headless Host
Abrir la conexion al host headless y verificar que la sala este disponible para recibir eventos en tiempo real.

In [5]:
def start_host(token: str, room_name: str = "Host Persistente Notebook"):
    env = os.environ.copy()
    env["HAXBALL_TOKEN"] = token
    env["HAXBALL_ROOM_NAME"] = room_name

    process = subprocess.Popen(
        ["npm", "run", "start"],
        cwd=PROJECT_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
    )
    return process


def tail_process_output(process, seconds: int = 8):
    start = time.time()
    lines = []
    while time.time() - start < seconds:
        line = process.stdout.readline()
        if not line:
            break
        lines.append(line.rstrip())
    return lines

print("Define tu token y ejecuta estas lineas manualmente cuando quieras levantar la sala:")
print("token = 'TU_TOKEN'\nhost_process = start_host(token)\nprint(*tail_process_output(host_process), sep='\\n')")

Define tu token y ejecuta estas lineas manualmente cuando quieras levantar la sala:
token = 'TU_TOKEN'
host_process = start_host(token)
print(*tail_process_output(host_process), sep='\n')


In [8]:
token = 'TU_TOKEN'
host_process = start_host(token)
print(*tail_process_output(host_process), sep='\n')


> haxball-host-persistente@1.0.0 start
> node src/runner.js

browserType.launch: Executable doesn't exist at /home/andreschirinos/.cache/ms-playwright/chromium_headless_shell-1223/chrome-headless-shell-linux64/chrome-headless-shell
╔════════════════════════════════════════════════════════════╗
║ Looks like Playwright was just installed or updated.       ║
║ Please run the following command to download new browsers: ║
║                                                            ║
║     npx playwright install                                 ║
║                                                            ║
║ <3 Playwright Team                                         ║
╚════════════════════════════════════════════════════════════╝
    at main (/home/andreschirinos/Proyectos/haxball-host/src/runner.js:20:34)
    at Object.<anonymous> (/home/andreschirinos/Proyectos/haxball-host/src/runner.js:159:1) {
  name: 'Error'
}


## 3. Registrar eventos de partidos y jugadores
Capturar entradas y salidas de jugadores, cambios de equipo, goles, inicio y fin de partido.

In [6]:
def load_jsonl(file_path: Path):
    if not file_path.exists():
        return []
    rows = []
    for line in file_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line:
            rows.append(json.loads(line))
    return rows


events = load_jsonl(EVENTS_FILE)
matches = load_jsonl(MATCHES_FILE)
players = json.loads(PLAYERS_FILE.read_text(encoding="utf-8")) if PLAYERS_FILE.exists() else {}

print(f"Eventos: {len(events)}")
print(f"Partidos: {len(matches)}")
print(f"Jugadores unicos: {len(players)}")

Eventos: 0
Partidos: 0
Jugadores unicos: 0


## 4. Mapear equipos, cambios y estados del juego
Mantener una estructura de datos con el estado actual de cada jugador, su equipo y la composicion de cada lado.

In [7]:
team_state = {}

for event in events:
    event_type = event.get("type")
    if event_type in {"player_join", "team_change", "player_leave"}:
        player = event.get("player", {})
        player_id = player.get("id")
        if player_id is None:
            continue

        current = team_state.get(player_id, {
            "id": player_id,
            "name": player.get("name", "desconocido"),
            "team": "spec",
            "connected": False,
            "last_event_at": None,
        })

        current["name"] = player.get("name", current["name"])
        current["last_event_at"] = event.get("at")

        if event_type == "player_join":
            current["connected"] = True
            current["team"] = player.get("team", current["team"])
        elif event_type == "team_change":
            team_num = event.get("team", 0)
            current["team"] = "red" if team_num == 1 else "blue" if team_num == 2 else "spec"
        elif event_type == "player_leave":
            current["connected"] = False

        team_state[player_id] = current

players_state_df = pd.DataFrame(team_state.values()).sort_values(["team", "name"]) if team_state else pd.DataFrame()
players_state_df

""


## 5. Guardar resultados y estadisticas en almacenamiento persistente
Persistir partidos, marcadores, listas de jugadores por equipo y metadatos en archivos o base de datos.

In [ ]:
matches_df = pd.DataFrame(matches)

if not matches_df.empty:
    def winner(row):
        score = row.get("finalScore") or {}
        red = score.get("red", 0)
        blue = score.get("blue", 0)
        if red > blue:
            return "red"
        if blue > red:
            return "blue"
        return "draw"

    matches_df["winner"] = matches_df.apply(winner, axis=1)
    wins = matches_df["winner"].value_counts().rename_axis("team").reset_index(name="wins")
else:
    wins = pd.DataFrame(columns=["team", "wins"])

wins

## 6. Consultar y exportar el historial de partidos
Generar consultas para revisar partidos anteriores y exportar los datos en formatos como CSV o JSON.

In [ ]:
goals_df = pd.DataFrame([e for e in events if e.get("type") == "team_goal"])

if not goals_df.empty:
    goals_df = goals_df[["at", "team"]].tail(10)

export_dir = DATA_DIR / "exports"
export_dir.mkdir(exist_ok=True)

if not matches_df.empty:
    matches_df.to_csv(export_dir / "matches.csv", index=False)
    matches_df.to_json(export_dir / "matches.json", orient="records", force_ascii=False, indent=2)

if not players_state_df.empty:
    players_state_df.to_csv(export_dir / "players_state.csv", index=False)

print(f"Exportaciones en: {export_dir}")
print("Ultimos 10 goles:")
goals_df